In [1]:
import pandas as pd
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Concatenate
import numpy as np

### Prepare Data

In [2]:
# Load the dataset
df = pd.read_csv('df_start.tsv')

# Convert timestamps to pandas datetime format
df['timestamp'] = pd.to_datetime(df['timestamp'])

In [3]:
# Divide the day into 8 time bins
time_bins = pd.cut(df['timestamp'].dt.hour, bins=[0, 3, 6, 9, 12, 15, 18, 21, 24], labels=False, right=False)

# Add time bins to the dataframe
df['time_bin'] = time_bins

In [4]:
# Sort by user and timestamp to ensure the sequences are in the correct order
df.sort_values(by=['user_id', 'timestamp'], inplace=True)

In [5]:
# Encode your 'user_id' and 'app_name' columns
user_encoder = LabelEncoder()
app_encoder = LabelEncoder()

df['user_id_encoded'] = user_encoder.fit_transform(df['user_id'])
df['app_name_encoded'] = app_encoder.fit_transform(df['app_name'])

In [6]:
# Group by 'user_id' and create sequences of 'app_name_encoded' and 'time_bin'
grouped = df.groupby('user_id')
app_sequences = grouped['app_name_encoded'].apply(list)
time_bin_sequences = grouped['time_bin'].apply(list)

In [7]:
# Create labels (the next app in the sequence)
# The label for each sequence is the next app that was used
app_labels = [sequence[1:] for sequence in app_sequences]
app_sequences = [sequence[:-1] for sequence in app_sequences]  # Remove last app since it's now a label

In [8]:
# Pad the sequences so they have the same length
max_sequence_length = max(len(seq) for seq in app_sequences)
app_sequences_padded = pad_sequences(app_sequences, maxlen=max_sequence_length, padding='post')
time_bin_sequences_padded = pad_sequences(time_bin_sequences, maxlen=max_sequence_length, padding='post')

In [9]:
# Make sure the labels are the same length as the inputs
app_labels_padded = pad_sequences(app_labels, maxlen=max_sequence_length, padding='post')

In [10]:
# Trim the last timestep from labels since we're predicting the next app (and there's no "next" for the last one)
#app_sequences_padded = app_sequences_padded[:, :-1]
app_labels_padded = app_labels_padded[:, -1] 
#app_labels_padded = app_labels_padded[:, :-1]
#time_bin_sequences_padded = time_bin_sequences_padded[:, :-1]

In [11]:
#max_sequence_length = max_sequence_length - 1

In [11]:
# Make sure your inputs and outputs are the same length
assert len(app_sequences_padded) == len(time_bin_sequences_padded) == len(app_labels_padded), "Mismatch in input and output lengths."
#assert len(app_sequences_padded[0]) == len(time_bin_sequences_padded[0]) == len(app_labels_padded[0]), "Mismatch in input and output dimension."

In [12]:
# Reshape user_ids to be compatible with the model input
user_ids = df['user_id_encoded'].unique()
user_ids_input = np.repeat(user_ids, max_sequence_length).reshape(-1, max_sequence_length)

In [13]:
assert len(app_sequences_padded[0]) == len(time_bin_sequences_padded[0]) == len(app_labels_padded[0]) == len(user_ids_input[0]), "Mismatch in input and output dimension."

TypeError: object of type 'numpy.int32' has no len()

### Build the Model

In [14]:
# Model building
num_users = len(user_encoder.classes_)
num_apps = len(app_encoder.classes_)

In [15]:
user_input = Input(shape=(max_sequence_length,), name='user_input')
app_input = Input(shape=(max_sequence_length,), name='app_input')
time_bin_input = Input(shape=(max_sequence_length,), name='time_bin_input')

In [16]:
user_embedding = Embedding(num_users, 50, input_length=max_sequence_length, name='user_embedding')(user_input)
app_embedding = Embedding(num_apps, 50, input_length=max_sequence_length, name='app_embedding')(app_input)
time_bin_embedding = Embedding(8, 50, input_length=max_sequence_length, name='time_bin_embedding')(time_bin_input)


In [17]:
# LSTM layer
lstm_out = LSTM(50)(Concatenate()([user_embedding, app_embedding, time_bin_embedding]))


In [18]:
# Output layer
output = Dense(num_apps, activation='softmax')(lstm_out)

In [19]:
model = Model(inputs=[user_input, app_input, time_bin_input], outputs=output)

In [20]:
# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

### Train the Model

#### Train, test, validation split

In [21]:
from sklearn.model_selection import train_test_split

# Assuming that user_ids_input, app_sequences_padded, and time_bin_sequences_padded
# are all lists or arrays with the same number of elements:
X = list(zip(user_ids_input, app_sequences_padded, time_bin_sequences_padded))

# Now X is a single list where each element is a tuple containing all inputs for a given sample
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    app_labels_padded, 
    test_size=0.2, 
    random_state=42
)

# Split the training data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_train, 
    y_train, 
    test_size=0.1, 
    random_state=42
)

# Now, before training, you would unpack the tuples in X_train, X_val, and X_test
# back into separate arrays so they can be input into the model
train_user_ids_input, train_app_sequences_padded, train_time_bin_sequences_padded = zip(*X_train)
val_user_ids_input, val_app_sequences_padded, val_time_bin_sequences_padded = zip(*X_val)
test_user_ids_input, test_app_sequences_padded, test_time_bin_sequences_padded = zip(*X_test)


In [41]:
print("User 0, first 10 apps in the input: ")
print(X_train)

print("User 0, first 10 apps in the label (shifted by 1): ")
print(y_train)

print(y_test)

User 0, first 10 apps in the input: 
[(array([247, 247, 247, ..., 247, 247, 247], dtype=int64), array([72, 23, 23, ...,  0,  0,  0]), array([4, 4, 4, ..., 0, 0, 0])), (array([201, 201, 201, ..., 201, 201, 201], dtype=int64), array([56, 32, 22, ...,  0,  0,  0]), array([0, 0, 0, ..., 0, 0, 0])), (array([205, 205, 205, ..., 205, 205, 205], dtype=int64), array([32, 22, 59, ...,  0,  0,  0]), array([0, 1, 3, ..., 0, 0, 0])), (array([100, 100, 100, ..., 100, 100, 100], dtype=int64), array([64, 22, 64, ...,  0,  0,  0]), array([6, 6, 6, ..., 0, 0, 0])), (array([289, 289, 289, ..., 289, 289, 289], dtype=int64), array([71, 71, 65, ...,  0,  0,  0]), array([6, 6, 6, ..., 0, 0, 0])), (array([86, 86, 86, ..., 86, 86, 86], dtype=int64), array([24, 28, 24, ...,  0,  0,  0]), array([7, 7, 7, ..., 0, 0, 0])), (array([285, 285, 285, ..., 285, 285, 285], dtype=int64), array([67, 67, 53, ...,  0,  0,  0]), array([4, 4, 4, ..., 0, 0, 0])), (array([223, 223, 223, ..., 223, 223, 223], dtype=int64), array([

In [46]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import ModelCheckpoint

# Assuming that your data is in tuples because of the zip operation, convert them to numpy arrays
train_user_ids = np.array(train_user_ids_input)
train_app_sequences = np.array(train_app_sequences_padded)
train_time_bin_sequences = np.array(train_time_bin_sequences_padded)
val_user_ids = np.array(val_user_ids_input)
val_app_sequences = np.array(val_app_sequences_padded)
val_time_bin_sequences = np.array(val_time_bin_sequences_padded)

# Define the checkpoint directory to save the models
checkpoint_path = "checkpoints/model_epoch_{epoch:02d}.hdf5"

# Create a ModelCheckpoint callback that saves the model's weights at the end of every epoch
checkpoint_callback = ModelCheckpoint(
    filepath=checkpoint_path,
    save_weights_only=False,  # Set to True if you just want to save the weights, not the whole model
    save_best_only=False,     # Set to True if you only want to save when there is an improvement in validation loss
    verbose=1,                # Logs out when files are being written
    save_freq='epoch'                  # Save every epoch
)

# Now you can fit the model
history = model.fit(
    [train_user_ids, train_app_sequences, train_time_bin_sequences],
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=([val_user_ids, val_app_sequences, val_time_bin_sequences], y_val),
    callbacks=[checkpoint_callback]  # Add the callback to the list of callbacks
)

# Plot training & validation loss
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss over epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()


Epoch 1/10
7/7 [==============================] - ETA: 0s - loss: 4.3793 - accuracy: 0.5598  
Epoch 1: saving model to checkpoints\model_epoch_01.hdf5
7/7 [==============================] - 1321s 189s/step - loss: 4.3793 - accuracy: 0.5598 - val_loss: 4.2127 - val_accuracy: 1.0000
Epoch 2/10


c:\Users\Brin\AppData\Local\r-miniconda\envs\next_app_prediction\lib\site-packages\keras\src\engine\training.py:3079: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


7/7 [==============================] - ETA: 0s - loss: 3.9840 - accuracy: 0.9952  
Epoch 2: saving model to checkpoints\model_epoch_02.hdf5
7/7 [==============================] - 1806s 260s/step - loss: 3.9840 - accuracy: 0.9952 - val_loss: 3.4271 - val_accuracy: 1.0000
Epoch 3/10
2/7 [=======>......................] - ETA: 25:26 - loss: 3.2680 - accuracy: 1.0000

### Evaluate on Test Set

In [23]:
# Assuming test_user_ids, test_app_sequences, and test_time_bin_sequences are lists
test_user_ids = np.array(test_user_ids_input)
test_app_sequences = np.array(test_app_sequences_padded)
test_time_bin_sequences = np.array(test_time_bin_sequences_padded)

# Evaluate the model on the test set
test_loss, test_accuracy = model.evaluate(
    [test_user_ids, test_app_sequences, test_time_bin_sequences],
    y_test
)

# Output the results
print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")


2/2 [==============================] - 30s 18s/step - loss: 0.0256 - accuracy: 1.0000
Test Loss: 0.025567030534148216
Test Accuracy: 1.0
